# 01 — Calibração: reproduzir a referência local numa A100

**Objetivo:** provar que a transferência para o Colab é limpa **antes** de concluir
qualquer coisa aqui. Não tire conclusão nenhuma até esta passar.

Referência local: **20,74 ± 0,33 W/m²** de RMSE de DHI no teste (3 sementes, RTX 2060
SUPER, entrada **não** padronizada). Banda de aceitação **20,74 ± 0,66** (2× o desvio
entre sementes).

> **Sobre a padronização ImageNet.** O modo imagem entregava ao DINOv2 um frame `[0,1]`
> cru enquanto o caminho de embedding padronizava — os dois discordavam, e isso era um
> defeito real. Mas corrigi-lo foi **medido e piorou**: 21,35 ± 0,22 contra 20,74 ± 0,33,
> +0,61 W/m² (~2,7σ). Provável confundidor: `backbone_lr` e `lr` foram escolhidos no
> regime não-padronizado, e padronizar muda a escala do gradiente em 4–8×. **O primeiro
> experimento que vale rodar aqui é re-afinar `backbone_lr` sob padronização**, não
> assumir que a correção é gratuita.


## 1. Runtime e GPU

**Antes de rodar:** `Runtime > Change runtime type > A100 GPU`, com *High-RAM* **desligado**
— o pico medido é 482 MiB no ViT-S e ~4 GB no ViT-B, e a variante de 80 GB custa +39% de
unidades por memória que não usamos.

Custo: A100-40GB ≈ 5,4 unidades/h, então 24 h ≈ 130 CU ≈ **26% da cota mensal do Pro+**
(500 CU). Confira a taxa real em *View resources*, no menu superior direito.


In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False).stdout)

## 2. Ambiente

O pacote exige CPython >= 3.14, que o Colab normalmente não traz — por isso o `uv`
provisiona um interpretador próprio. **A instalação do torch CUDA é obrigatória e
verificada:** o extra `allsky` fixa uma wheel de CPU, e um torch de CPU aqui invalida
a sessão inteira.

Esta é a única célula que não pode vir do `_colab_runner`: é ela que clona o repo onde
o runner mora.


In [ ]:
import os
import subprocess
import sys

REPO = "https://github.com/Bruno-Mascarenhas/micrometeorology.git"
BRANCH = "main"
WORKDIR = "/content/micrometeorology"

if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, WORKDIR], check=True)
subprocess.run(["pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.14"], cwd=WORKDIR, check=True)
subprocess.run(["uv", "venv", "--python", "3.14", ".venv"], cwd=WORKDIR, check=True)
subprocess.run(["uv", "sync", "--locked", "--extra", "allsky"], cwd=WORKDIR, check=True)
subprocess.run(
    [
        "uv",
        "pip",
        "install",
        "--python",
        ".venv/bin/python",
        "--reinstall",
        "--torch-backend",
        "cu130",
        "torch==2.13.0",
    ],
    cwd=WORKDIR,
    check=True,
)

PY = f"{WORKDIR}/.venv/bin/python"
os.environ["PATH"] = f"{WORKDIR}/.venv/bin:" + os.environ["PATH"]
sys.path.insert(0, f"{WORKDIR}/notebooks/colab")

verify = subprocess.run(
    [PY, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
    capture_output=True,
    text=True,
    check=False,
)
print(verify.stdout)
if "True" not in verify.stdout:
    raise RuntimeError("torch sem CUDA — pare e reinstale antes de treinar")

## 3. Dados e artefatos

`stage_bundle` copia o bundle para o SSD local, desempacota e roda `validate-dataset`.
Os três passos importam: treinar de `/content/drive` é FUSE e a leitura fria domina a
época, e um bundle truncado treinaria em silêncio sem a validação.

`ARTIFACTS` no Drive é o que sobrevive à sessão — o timeout por inatividade do Colab só
conta **quando a execução termina**, e todo run para por early stopping na época ~20.


In [ ]:
import os

import _colab_runner as runner
from google.colab import drive

BUNDLE = "/content/drive/MyDrive/labmim/allsky-mm/bundle.tar.gz"
DATA = "/content/allsky-mm"
ARTIFACTS = "/content/drive/MyDrive/labmim/runs/allsky-mm"

drive.mount("/content/drive")
os.makedirs(ARTIFACTS, exist_ok=True)
ROOT = runner.stage_bundle(BUNDLE, DATA, python=PY)

## 4. Hardware e ajustes que dependem dele

`bf16` existe em toda GPU do Colab menos a T4 (Turing). O código local roda `fp16`
porque a 2060 não tem alternativa; aqui a escolha é automática.

O probe roda no interpretador do venv, que é onde o torch com CUDA está instalado.


In [ ]:
import json

probe = subprocess.run(
    [
        PY,
        "-c",
        'import json, sys; sys.path.insert(0, "' + WORKDIR + '/notebooks/colab"); '
        "import _colab_runner as r; print(json.dumps(r.probe_accelerator()))",
    ],
    capture_output=True,
    text=True,
    check=True,
)
HW = json.loads(probe.stdout.strip().splitlines()[-1])
AMP_DTYPE = HW["amp_dtype"]
WORKERS = min(8, HW["cpus"])
print(HW)
print(f"amp={AMP_DTYPE}  workers={WORKERS}")

## 5. Calibração — 3 sementes

Idêntico ao vencedor local exceto `amp.dtype` e o caminho dos dados. É um controle:
nada aqui é para ajustar.


In [ ]:
from pathlib import Path

CFG = Path(WORKDIR) / "configs/allsky/experiments/colab"
OUT = Path("/content/out")
rows = []

for seed in (42, 43, 44):
    config = runner.write_config(
        CFG / f"calib_s{seed}.yaml",
        extends=["../_base.yaml", "../../models/image_only.yaml"],
        name=f"calib_s{seed}",
        output_dir=str(OUT / f"calib_s{seed}"),
        seed=seed,
        data_root=ROOT,
        model={"backbone_frozen": False, "unfreeze_last_n": 12, "image_size": 224},
        train={
            "backbone_lr": 1e-5,
            "epochs": 40,
            "batch_size": 64,
            "num_workers": WORKERS,
            "amp": {"enabled": True, "dtype": AMP_DTYPE},
        },
        targets=runner.DHI_ONLY_TARGETS,
        note="CONTROLE. Banda de aceitacao 20,74 +- 0,64 W/m2.",
    )
    row = runner.run_experiment(config)
    print(row.get("status"), row.get("rmse"), row.get("wall_seconds"))
    print(runner.archive(str(OUT / f"calib_s{seed}"), ARTIFACTS, config=config))
    rows.append(row)

runner.summarise(rows)

## 6. O portão

Fora da banda **para cima**: diagnostique antes de seguir — torch de CPU, bundle
diferente, ou `bf16` mudando o resultado (repita com `fp16`).
Fora da banda **para baixo**: é o ganho da padronização. Adote a nova média como
referência dos notebooks 02 e 03.


In [ ]:
import json

import numpy as np
import pandas as pd

ok = [r for r in rows if r.get("status") == "ok"]
# Sem amostra nao ha calibracao: `rmse.mean()` e NaN, as duas comparacoes contra
# NaN sao False e o ramo final publicava "transferencia limpa" com zero braco
# concluido — exatamente a conclusao que este notebook existe para recusar.
if len(ok) < 3:
    raise RuntimeError(
        f"{len(rows) - len(ok)} de {len(rows)} bracos falharam; "
        "nenhuma calibracao medida — nao siga para 02/03"
    )
rmse = np.array([r["rmse"] for r in ok])
if not np.isfinite(rmse).all():
    raise RuntimeError(f"RMSE nao finito em {int((~np.isfinite(rmse)).sum())} braco(s)")
lo, hi = 20.74 - 0.64, 20.74 + 0.64
print(f"RMSE {rmse.mean():.2f} +- {rmse.std(ddof=1):.2f} (n={len(rmse)})")
print(f"MBE  {np.mean([r['mbe'] for r in ok]):+.2f}")
if rmse.mean() < lo:
    print(f"ABAIXO da banda por {lo - rmse.mean():.2f} W/m2 — provavel ganho da padronizacao;")
    print("adote esta media como a nova referencia.")
elif rmse.mean() > hi:
    print("ACIMA da banda — diagnostique antes de seguir para 02/03.")
else:
    print("dentro da banda: transferencia limpa.")

pd.DataFrame(rows).to_csv(f"{ARTIFACTS}/calibracao.csv", index=False)
with open(f"{ARTIFACTS}/hardware.json", "w") as handle:
    json.dump(HW, handle, indent=2)
print("referencia gravada em", ARTIFACTS)